In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
df = pd.read_csv(r'/content/data_for_predictions.csv')
df.drop(columns=["Unnamed: 0"], inplace=True)
df.head()

,id,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,forecast_price_pow_off_peak,...,months_modif_prod,months_renewal,channel_MISSING,channel_ewpakwlliwisiwduibdlfmalxowmwpci,channel_foosdfpfkusacimwkcsosbicdxkicaua,channel_lmkebamcaaclubfxadlmueccxoimlema,channel_usilxuppasemubllopkaafesmlibmsdf,origin_up_kamkkxfxxuwbdslkwifmmcsiusiuosws,origin_up_ldkssxwpmemidmecebumciepifcamkci,origin_up_lxidpiddsbxsbosboudacockeimpuepw
0,24011ae4ebbe3035111d65fa7c15bc57,0.000000,4.739944,0.000000,0.000000,0.0,0.444045,0.114481,0.098142,40.606701,...,2,6,0,0,1,0,0,0,0,1
1,d29c2c54acc38ff3c0614d0a653813dd,3.668479,0.000000,0.000000,2.280920,0.0,1.237292,0.145711,0.000000,44.311378,...,76,4,1,0,0,0,0,1,0,0
2,764c75f661154dac3a6c254cd082ea7d,2.736397,0.000000,0.000000,1.689841,0.0,1.599009,0.165794,0.087899,44.311378,...,68,8,0,0,1,0,0,1,0,0
3,bba03439a292a1e166f80264c16191cb,3.200029,0.000000,0.000000,2.382089,0.0,1.318689,0.146694,0.000000,44.311378,...,69,9,0,0,0,1,0,1,0,0
4,149d57cf92fc41cf94415803a877cb4b,3.646011,0.000000,2.721811,2.650065,0.0,2.122969,0.116900,0.100015,40.606701,...,71,9,1,0,0,0,0,1,0,0


In [ ]:
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14606 entries, 0 to 14605
Data columns (total 63 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   id                                          14606 non-null  object 
 1   cons_12m                                    14606 non-null  float64
 2   cons_gas_12m                                14606 non-null  float64
 3   cons_last_month                             14606 non-null  float64
 4   forecast_cons_12m                           14606 non-null  float64
 5   forecast_discount_energy                    14606 non-null  float64
 6   forecast_meter_rent_12m                     14606 non-null  float64
 7   forecast_price_energy_off_peak              14606 non-null  float64
 8   forecast_price_energy_peak                  14606 non-null  float64
 9   forecast_price_pow_off_peak                 14606 non-null  float64
 10  has_gas   

In [ ]:
train_df = df.copy()

y = df['churn'] # target
X = df.drop(columns=['id', 'churn']) # features
print(X.shape)
print(y.shape)

(14606, 61)
(14606,)


In [ ]:
# train on 80% of data, test accuracy on 20% of data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# heavily imbalanced
df.churn.value_counts()

,count
churn,
0,13187
1,1419


In [ ]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [ ]:
y_pred = model.predict(X_test)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# standard Random Forest Model does poorly when trying to identify churners (0.06 recall)


Classification Report:

              precision    recall  f1-score   support

           0       0.90      1.00      0.95      2617
           1       0.89      0.06      0.10       305

    accuracy                           0.90      2922
   macro avg       0.90      0.53      0.53      2922
weighted avg       0.90      0.90      0.86      2922



In [ ]:
tn, fp, fn, tp = metrics.confusion_matrix(y_test, y_pred).ravel()

print(f"True positives: {tp}")
print(f"False positives: {fp}")
print(f"True negatives: {tn}")
print(f"False negatives: {fn}")

True positives: 17
False positives: 2
True negatives: 2615
False negatives: 288


In [ ]:
# use SMOTE to oversample the minority class + xgbclassifier for our model
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [ ]:
scale_weight = 2617 / 305

model = XGBClassifier(
    scale_pos_weight=scale_weight, # used for imbalanced datasets, penalizes mistakes for minority class
    max_depth=5,
    learning_rate=0.01
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [ ]:
# true negatives, false positives, false negatives, true positives
tn, fp, fn, tp = metrics.confusion_matrix(y_test, y_pred).ravel()

In [ ]:
print(f"True positives: {tp}")
print(f"False positives: {fp}")
print(f"True negatives: {tn}")
print(f"False negatives: {fn}\n")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

True positives: 143
False positives: 609
True negatives: 2008
False negatives: 162


Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.77      0.84      2617
           1       0.19      0.47      0.27       305

    accuracy                           0.74      2922
   macro avg       0.56      0.62      0.55      2922
weighted avg       0.85      0.74      0.78      2922



In [ ]:
smote = SMOTE(sampling_strategy=0.2, random_state=42, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [ ]:
model.fit(X_train_resampled, y_train_resampled)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.01, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
y_pred_resampled = model.predict(X_test)

In [ ]:
tn, fp, fn, tp = metrics.confusion_matrix(y_test, y_pred_resampled).ravel()

In [ ]:
print(f"True positives: {tp}")
print(f"False positives: {fp}")
print(f"True negatives: {tn}")
print(f"False negatives: {fn}\n")

print(classification_report(y_test, y_pred_resampled))

# best recall score for churners, but lots of false positives and worse f1-score
# when trying to correctly identify churners, it may be worth it to use this model

True positives: 230
False positives: 1595
True negatives: 1022
False negatives: 75

              precision    recall  f1-score   support

           0       0.93      0.39      0.55      2617
           1       0.13      0.75      0.22       305

    accuracy                           0.43      2922
   macro avg       0.53      0.57      0.38      2922
weighted avg       0.85      0.43      0.52      2922



In [ ]:
y_train_resampled.value_counts()

,count
churn,
0,10570
1,2114


In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

model = XGBClassifier(random_state=42)

In [ ]:
grid_search = GridSearchCV(estimator=model,
                           param_grid=param_grid,
                           scoring='f1', # Optimize for F1 score
                           cv=3,
                           n_jobs=-1,
                           verbose=1
                           )

grid_search.fit(X_train_resampled, y_train_resampled)

model = grid_search.best_estimator_

Fitting 3 folds for each of 243 candidates, totalling 729 fits


In [ ]:
print(grid_search.best_params_)
print(grid_search.best_score_)

y_pred_best_f1 = model.predict(X_test)

print("\nClassification Report for Best F1 Model (XGBoost with GridSearchCV):\n")
print(classification_report(y_test, y_pred_best_f1))

tn_f1, fp_f1, fn_f1, tp_f1 = metrics.confusion_matrix(y_test, y_pred_best_f1).ravel()
print(f"\nTrue positives: {tp_f1}")
print(f"False positives: {fp_f1}")
print(f"True negatives: {tn_f1}")
print(f"False negatives: {fn_f1}")

{'colsample_bytree': 0.6, 'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.6}
0.5870794631387785

Classification Report for Best F1 Model (XGBoost with GridSearchCV):

              precision    recall  f1-score   support

           0       0.91      0.99      0.95      2617
           1       0.62      0.16      0.25       305

    accuracy                           0.90      2922
   macro avg       0.76      0.57      0.60      2922
weighted avg       0.88      0.90      0.87      2922


True positives: 48
False positives: 30
True negatives: 2587
False negatives: 257


In [ ]:
grid_search = GridSearchCV(estimator=model,
                           param_grid=param_grid,
                           scoring='recall', # Optimize for recall score
                           cv=3,
                           n_jobs=-1,
                           verbose=1
                           )

grid_search.fit(X_train_resampled, y_train_resampled)

model = grid_search.best_estimator_

Fitting 3 folds for each of 243 candidates, totalling 729 fits


In [ ]:
print(grid_search.best_params_)
print(grid_search.best_score_)

y_pred_best_recall = model.predict(X_test)

print("\nClassification Report for Best Recall Model (XGBoost with GridSearchCV):\n")
print(classification_report(y_test, y_pred_best_recall))

tn_r, fp_r, fn_r, tp_r = metrics.confusion_matrix(y_test, y_pred_best_recall).ravel()
print(f"\nTrue positives (Best Recall Model): {tp_r}")
print(f"False positives (Best Recall Model): {fp_r}")
print(f"True negatives (Best Recall Model): {tn_r}")
print(f"False negatives (Best Recall Model): {fn_r}")

{'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.6}
0.5107034440146142

Classification Report for Best Recall Model (XGBoost with GridSearchCV):

              precision    recall  f1-score   support

           0       0.91      0.99      0.95      2617
           1       0.59      0.16      0.25       305

    accuracy                           0.90      2922
   macro avg       0.75      0.57      0.60      2922
weighted avg       0.88      0.90      0.87      2922


True positives (Best Recall Model): 49
False positives (Best Recall Model): 34
True negatives (Best Recall Model): 2583
False negatives (Best Recall Model): 256
